# Phase 3 processed-data quality inspection
Run from the repository root. This notebook only inspects verified local Parquet outputs; no API or modeling is involved.

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent  # nbconvert may start kernels in notebooks/
assert (root / 'pyproject.toml').exists(), 'Run from the repository root'
processed = root / 'data/processed/openalex_stratified/stratified_20261019T155909Z'
assert processed.is_dir(), 'Run openalex-preprocess from the repository root first'
names = ['works', 'work_topics', 'authorships', 'work_institutions', 'work_countries', 'references', 'category_membership', 'population_year_counts']
tables = {name: pq.read_table(processed / f'{name}.parquet').to_pandas() for name in names}
pd.DataFrame([{'table': name, 'rows': len(frame), 'columns': len(frame.columns)} for name, frame in tables.items()])

In [ ]:
for name in names:
    print(f'\n{name}: {len(tables[name]):,} rows')
    print(pq.read_schema(processed / f'{name}.parquet'))

In [ ]:
works = tables['works']
assert works.work_id.is_unique
display(works[['title', 'doi', 'abstract', 'fwci', 'language', 'is_oa', 'oa_status']].isna().agg(['sum', 'mean']))
display(works.groupby('publication_year', dropna=False).size().rename('sample_works').to_frame())
display(works.groupby('work_type', dropna=False).size().rename('sample_works').to_frame())
display(works.groupby('work_type', dropna=False)['has_fwci'].agg(['size', 'sum', 'mean']))

In [ ]:
membership = tables['category_membership']
display(membership.groupby(['analytical_layer', 'category']).size().rename('matched_works').to_frame())
assert not membership.duplicated(['work_id', 'analytical_layer', 'category']).any()
# Joining an overlapping membership bridge directly multiplies work rows.
naive = works[['work_id']].merge(membership[['work_id', 'category']], on='work_id', how='left')
safe = works[['work_id']].merge(membership.groupby('work_id').size().rename('category_count'), on='work_id', how='left')
safe['category_count'] = safe['category_count'].fillna(0).astype('int64')
display(pd.DataFrame({'join': ['direct (unsafe for work counts)', 'aggregated (safe)'], 'rows': [len(naive), len(safe)], 'unique_works': [naive.work_id.nunique(), safe.work_id.nunique()]}))
assert len(safe) == len(works)